[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NASA-Gradient-Descent-Farmers/NASA-warm-up/blob/sant-calibration/Notebooks/initial_exploration.ipynb)

# Initial exploration: from raw NASA data to a field-season table

* **Author:** Santiago (Gradient Descent Farmers)
* **Branch:** `sant-calibration`

Rotation decision tools almost always work on one table: **one row per field per season**, with the crop, the previous crop, what the soil carried over, what the weather did, and an outcome. This notebook looks at what each data source in `src/data_ingestion/` actually returns, then reshapes it into that table so the gaps are visible.

| Source | Raw shape | Login |
|---|---|---|
| NASA POWER | Daily time series at a point | No |
| GLDAS soils | One static row per location | No |
| IGBP-DIS soils | One static row per location | Earthdata |
| SMAP L4 | 3-hourly time series at a 9 km pixel | Earthdata |

Sections 1-4 run without a login. SMAP and IGBP are behind flags in section 5.

## 0. Setup

In Colab this clones the repo (this branch) and installs `earthaccess`. Locally it just finds the repo root.

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/NASA-Gradient-Descent-Farmers/NASA-warm-up.git"
BRANCH = "sant-calibration"

if "google.colab" in sys.modules:
    if not Path("/content/NASA-warm-up").exists():
        !git clone -q --branch {BRANCH} {REPO_URL} /content/NASA-warm-up
    !pip install -q "earthaccess>=0.11"
    ROOT = Path("/content/NASA-warm-up")
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data_ingestion").is_dir())

os.chdir(ROOT)  # the ingestion scripts cache downloads under data/ relative to the repo root
sys.path.insert(0, str(ROOT))
(ROOT / "data").mkdir(exist_ok=True)
print("Repo root:", ROOT)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.data_ingestion import power
from src.data_ingestion.common import SAMPLE_SITES

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

### Site and season

Default site: **Celaya, Guanajuato** (Bajío). Change `SITE` to any `(name, lat, lon)`; `SAMPLE_SITES` has the team's five, and Isaac's WOFOST runs use Texcoco (`("Texcoco", 19.53, -98.85)`).

The rainfed spring-summer cycle (*Primavera-Verano, temporal*) runs roughly May to October, so that's the season window. Adjust it for irrigated or autumn-winter crops.

In [ ]:
SITE = SAMPLE_SITES[2]           # ("Celaya, GTO (Bajio, MX)", 20.52, -100.82)
SITE_NAME, LAT, LON = SITE

FIRST_YEAR, LAST_YEAR = 2001, 2025
SEASON_MONTHS = (5, 10)          # May-October, rainfed spring-summer cycle
CANICULA_MONTHS = (7, 8)         # mid-summer drought window

print(SITE)

## 1. NASA POWER: what daily weather looks like

One request returns every day for the point. Columns are variables, the index is the date. POWER's grid cell is ~50 km, so this is the weather of the region, not the field.

In [ ]:
payload = power.fetch_daily_point(LAT, LON, f"{FIRST_YEAR}0101", f"{LAST_YEAR}1231")
weather = power.to_dataframe(payload)
units = {k: v["units"] for k, v in payload["parameters"].items()}

print("Shape:", weather.shape, "| from", weather.index.min().date(), "to", weather.index.max().date())
print("Units:", units)
weather.head()

In [ ]:
# Data quality: missing values per column and gaps in the date index
print("Missing values per column:")
print(weather.isna().sum())
full_range = pd.date_range(weather.index.min(), weather.index.max(), freq="D")
print("\nMissing dates:", len(full_range.difference(weather.index)))
weather.describe().round(2)

In [ ]:
# One year, to see the seasonality the season features have to capture
year = 2024
w = weather.loc[str(year)]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
ax1.bar(w.index, w["PRECTOTCORR"], width=1, color="tab:blue")
ax1.set_ylabel("Rain (mm/day)")
ax1.set_title(f"{SITE_NAME}, {year}")
ax2.fill_between(w.index, w["T2M_MIN"], w["T2M_MAX"], alpha=0.3, color="tab:red", label="Tmin-Tmax")
ax2.plot(w.index, w["T2M"], color="tab:red", lw=1, label="Tmean")
ax2.axhline(0, color="k", lw=0.5)
ax2.set_ylabel("°C")
ax2.legend(loc="upper right")
for ax in (ax1, ax2):
    ax.axvspan(pd.Timestamp(year, SEASON_MONTHS[0], 1), pd.Timestamp(year, SEASON_MONTHS[1], 31), color="green", alpha=0.06)
fig.tight_layout()
plt.show()

## 2. From days to seasons

A crop decision is made once a season, so daily weather has to be summarized into one row per year. These are the features that matter agronomically:

| Feature | Why |
|---|---|
| `rain_season_mm` | Total water for a rainfed crop |
| `rain_canicula_mm` | July-August rain; a dry canícula hits maize at flowering |
| `longest_dry_spell_days` | Timing matters more than totals (consecutive days < 1 mm) |
| `onset_doy` | First day from May 1 with ≥ 20 mm over 5 days: when rainfed sowing is possible (same rule as `wofost_rotations.py`) |
| `gdd_maize` | Growing degree days, base 10 °C, capped at 30 °C: heat available to develop the crop |
| `frost_nights`, `heat_days` | Nights with Tmin ≤ 0 °C and days with Tmax ≥ 35 °C during the season |
| `year_class` | Dry / normal / wet by terciles of season rain |

In [ ]:
def longest_run(mask):
    """Longest run of consecutive True values."""
    best = run = 0
    for value in mask:
        run = run + 1 if value else 0
        best = max(best, run)
    return best


def rain_onset(rain, start, window=5, threshold=20.0):
    """Day of year of the first day from `start` with >= threshold mm over the previous `window` days."""
    rolling = rain.rolling(window, min_periods=window).sum()
    hits = rolling[(rolling.index >= start) & (rolling >= threshold)]
    return hits.index[0].dayofyear if len(hits) else np.nan


def season_features(daily, months=SEASON_MONTHS, canicula=CANICULA_MONTHS):
    """One row per year of season-level weather features from POWER daily data."""
    rows = []
    for year, d in daily.groupby(daily.index.year):
        s = d[(d.index.month >= months[0]) & (d.index.month <= months[1])]
        if len(s) < 150:  # incomplete season
            continue
        tmean = ((s["T2M_MAX"].clip(upper=30) + s["T2M_MIN"].clip(lower=10)) / 2)
        rows.append({
            "year": year,
            "rain_season_mm": s["PRECTOTCORR"].sum(),
            "rain_canicula_mm": s.loc[s.index.month.isin(range(canicula[0], canicula[1] + 1)), "PRECTOTCORR"].sum(),
            "longest_dry_spell_days": longest_run(s["PRECTOTCORR"] < 1.0),
            "onset_doy": rain_onset(d["PRECTOTCORR"], pd.Timestamp(year, months[0], 1)),
            "gdd_maize": (tmean - 10).clip(lower=0).sum(),
            "frost_nights": int((s["T2M_MIN"] <= 0).sum()),
            "heat_days": int((s["T2M_MAX"] >= 35).sum()),
            "tmean_season_c": s["T2M"].mean(),
            "radiation_mj_m2": s["ALLSKY_SFC_SW_DWN"].sum(),
        })
    seasons = pd.DataFrame(rows).set_index("year")
    seasons["year_class"] = pd.qcut(seasons["rain_season_mm"], 3, labels=["dry", "normal", "wet"])
    return seasons


seasons = season_features(weather)
print("Shape:", seasons.shape, "-> one row per season, down from", len(weather), "days")
seasons.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
colors = seasons["year_class"].map({"dry": "tab:orange", "normal": "tab:gray", "wet": "tab:blue"})

axes[0].bar(seasons.index, seasons["rain_season_mm"], color=colors)
axes[0].set(title="Season rain (May-Oct)", ylabel="mm")

axes[1].scatter(seasons["rain_season_mm"], seasons["longest_dry_spell_days"], c=colors)
for year, row in seasons.iterrows():
    axes[1].annotate(str(year)[2:], (row["rain_season_mm"], row["longest_dry_spell_days"]), fontsize=7)
axes[1].set(title="Total rain vs. longest dry spell", xlabel="mm", ylabel="days")

axes[2].plot(seasons.index, seasons["gdd_maize"], "o-")
axes[2].set(title="Growing degree days (maize)", ylabel="°C·day")
fig.tight_layout()
plt.show()

seasons.drop(columns="year_class").corr().round(2)

**Questions to look at here:** Are there wet years with long dry spells (rain arrives in a few big events)? Is there a trend in season rain or temperature over 25 years? How many truly independent seasons do we have? (That number, not the row count of any table built from it, limits what a model can learn.)

## 3. GLDAS soils: one static row

Soil texture changes over centuries, so each location gets one row. It joins onto every season of that site. The 0.25° grid cell is ~25 km: this is the region's dominant soil, not a field measurement.

In [ ]:
from src.data_ingestion import gldas_soils

gldas_soils.download()  # < 2 MB, cached in data/gldas/
soil = gldas_soils.extract_points([SITE])
soil

## 4. The field-season table, and what's missing

Combining the pieces: candidate crops × seasons, with weather and soil joined on. The last columns are what a rotation tool also needs and **none of our sources provide**:

* `crop`, `previous_crop`: what was actually grown on a field each year. The US has this as the USDA Cropland Data Layer; Mexico only has municipal totals (SIAP).
* `soil_water_at_sowing_mm`, `mineral_n_at_sowing_kg_ha`: carryover state. SMAP root-zone moisture can estimate the water part (section 5); nitrogen needs a simulator or soil tests.
* `yield_t_ha` (or soil carbon, water use): the outcome. It comes from trials, satellite yield proxies (peak NDVI), or a simulator like Isaac's WOFOST runs.

In [ ]:
CROPS = ["maize", "bean", "sorghum", "barley"]  # placeholder candidates for the Bajío

table = (
    seasons.reset_index()
    .merge(pd.DataFrame({"crop": CROPS}), how="cross")
    .assign(site=SITE_NAME)
    .merge(soil.reset_index().drop(columns=["lat", "lon"]), on="site", how="left")
    .assign(previous_crop=pd.NA, soil_water_at_sowing_mm=np.nan, mineral_n_at_sowing_kg_ha=np.nan, yield_t_ha=np.nan)
    .set_index(["site", "year", "crop"])
)
print("Shape:", table.shape)
print("Columns with no data source yet:", table.columns[table.isna().all()].tolist())
table.head(8)

## 5. Optional: Earthdata sources (SMAP and IGBP)

Both need a free account at [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov/users/new). Set the flags to `True` and log in when prompted.

* **IGBP-DIS** adds field capacity, wilting point and soil N (0-100 cm). The first download is ~220 MB.
* **SMAP L4** is 3-hourly, 8 values a day. A week takes a minute or two through OPeNDAP, so start small. Note: SMAP is a model with satellite data assimilated, not a direct measurement, and NSIDC flagged a geolocation issue from 14 May to 28 July 2026.

In [ ]:
RUN_IGBP = False
RUN_SMAP = False

if RUN_IGBP or RUN_SMAP:
    import earthaccess
    auth = earthaccess.login(persist=True)  # tries env vars and ~/.netrc, then prompts
    print("Authenticated:", auth.authenticated)

In [ ]:
if RUN_IGBP:
    from src.data_ingestion import igbp_soils

    igbp_soils.download(igbp_soils.search_granules())
    igbp = igbp_soils.extract_points([SITE])
    # Plant-available water: what the soil can hold for roots between field capacity and wilting point.
    # Check units in data/igbp/comp/readme.txt (expected mm of water for 0-100 cm).
    display(igbp)

In [ ]:
SMAP_START, SMAP_END = "2024-06-01", "2024-06-07"  # around rainfed sowing

if RUN_SMAP:
    from src.data_ingestion import smap

    granules = smap.search_granules(LAT, LON, SMAP_START, SMAP_END)
    print(len(granules), "granules (3-hourly)")
    sm, sm_units, pixel = smap.extract_point(granules, LAT, LON)
    print("Nearest 9 km pixel:", pixel, "| units:", sm_units)

    # 3-hourly UTC -> daily mean: the same day-to-season reshaping as POWER
    sm_daily = sm.resample("D").mean()
    display(sm_daily)

    ax = sm[["sm_surface", "sm_rootzone"]].plot(figsize=(12, 3), title=f"SMAP L4 soil moisture, {SITE_NAME}")
    ax.set_ylabel("m³/m³")
    plt.show()

## 6. Notes and next steps

* **Prediction vs. causal:** "What yield will this field get?" is prediction. "How much does a legume before maize help?" is a treatment effect, and farmers who rotate differ from those who don't. See *Precrop Payoffs* (Kluger & Lobell) in `Articles/`.
* **Effective sample size** is the number of independent seasons above, not the rows of the crossed table.
* **Ideas to try next:**
  1. MODIS NDVI for the same site: phenology (green-up, peak, senescence) as a yield proxy and a way to infer crop type where no crop map exists.
  2. SMAP root-zone moisture on the onset date as `soil_water_at_sowing_mm`.
  3. Compare these season features with Isaac's WOFOST outputs (`data/rotations/*.csv`) for Texcoco.